# Demo 10: your store, and a buyer that refuses well

**The brief:** a store the real `let_me_buy` program would accept, and a buyer
that shops from it. Scored out of 500, and **100 is a pass**.

This is weekly challenge 2, and the first step of your final project. On Monday
your store goes onto the shared course fork, next to everybody else's.

This notebook is the starting line. It shows the rules refusing a store that
looks fine, then hands you two cells to write. As shipped, the check scores
them 0 and says why.

Runs offline. No key, no wallet, no network.

In [1]:
# Setup. Works from anywhere inside the course checkout.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "pyproject.toml").exists():
    raise SystemExit(f"No course found above {Path.cwd()}. Open this inside your checkout.")
sys.path.insert(0, str(REPO_ROOT / "src"))

from bootcamp_agent.bonus import bonus
from bootcamp_agent.shipit.storefront import problems
from bootcamp_agent.weekly import week2_store  # noqa: F401 (registers the check)

print("ready")

ready


## 1. The rules are the program's, not ours

Every rule below is one the deployed program enforces, or a hazard its shape
creates. They are the Ship It track's rules, unchanged.

| Rule | Why |
|---|---|
| the store name starts with your GitHub handle | a store lives at the address made from its **name alone**. Two stores called `coffee` are one account, and the second one replaces the first |
| an address is 32 bytes | a made-up address can read fine and decode to 30 |
| a price is a whole number of the smallest unit | 1.5 USDC is `1500000`. A float sells at almost zero |
| decimals match the mint | USDC has 6. Get it wrong and every price is off by a thousand |
| the Telegram channel starts with `@` | it is where orders are sent. A bare username is one the order bot cannot reach, and every order is silently lost |
| at least 3 products | a buyer needs a choice to get right |

The last-but-one rule is not theory. A real store on mainnet had its channel
written without the `@`. It took orders for weeks and delivered none, and nothing
on chain could show it.

Watch the rules refuse a store that looks fine.

In [2]:
USDC = "EPjFWdd5AufqSSqeM2qN1xzybapC8G4wEGGkZwyTDt1v"

looks_fine = {
    "store": "coffee",
    "authority": "DemoAuthority1111111111111111111111111111",
    "telegram_channel_id": "coffeeorders",
    "products": [
        {"name": "Espresso", "price_raw": 1.5, "decimals": 9, "mint": USDC},
    ],
}

for problem in problems(looks_fine, handle="octocat"):
    print("-", problem)

- store: name it octocat-something. A store lives at the address derived from its NAME ALONE, so two people who both pick 'coffee' write to the same account and the second one silently replaces the first
- telegram_channel_id: 'coffeeorders' is a bare username. Write it with the @, like '@coffeeorders'. Without it the order bot cannot find the channel, so every order is silently swallowed: a real store lost every order it took this way
- authority: 'DemoAuthority1111111111111111111111111111' decodes to 30 bytes; a Solana address is 32. It reads like an address but cannot be one.
- products[0].price_raw: 1.5 is not an integer. A price counts the smallest unit, so 1.5 USDC is 1500000 -- a float here is how a store ends up selling at zero
- products[0].decimals: that mint has 6 decimals, not 9. Every price in this store is wrong by a factor of 1000


Five lines, and each one names the fix. None of them shows in a text editor.

## 2. Your store

Write your own. Your handle, a real address, whole-number prices, a channel with
the `@`, and at least three products.

**The authority** is the public address that owns the store. If you have a
wallet, use its address. If not, the line below prints a fresh one you can use
for now. It is only a public address: nothing is created and nothing is signed.
You can change it before Monday.

In [3]:
import os

from bootcamp_agent.shipit.borsh_store import b58encode

print("a fresh 32-byte address:", b58encode(os.urandom(32)))

a fresh 32-byte address: EvdBxL9huRc7EB3n8g7CqB5GrcLsoCcRrk4YMmnGqGHE


In [4]:
# ---------------------------------------------------------------------
# YOUR STORE. This cell runs as shipped, and the check below refuses it.
# Replace every REPLACE_ME, add products until there are at least three,
# and re-run until `problems` prints nothing.
# ---------------------------------------------------------------------
MY_GITHUB = "REPLACE_ME"

MY_STORE = {
    "store": "REPLACE_ME-coffee",
    "authority": "REPLACE_ME",
    "telegram_channel_id": "@REPLACE_ME",
    "products": [
        {"name": "Espresso", "price_raw": 1_000_000, "decimals": 6, "mint": USDC},
    ],
}

for problem in problems(MY_STORE, handle=MY_GITHUB):
    print("-", problem)

- authority: '_' is not a base58 character, so 'REPLACE_ME' is not an address


## 3. A buyer that refuses well

`plan_purchase(holdings, listing, request)` decides whether to buy. It is the
Ship It contract, unchanged.

| Argument | What it holds |
|---|---|
| `holdings` | `{mint: balance}`, whole numbers in the smallest unit |
| `listing` | a store, shaped like `MY_STORE` |
| `request` | `{"product": name, "quantity": whole number}` |

It returns `{"approved": True or False, "reason": "a sentence"}`.

**It refuses:**

- a product that is not on the menu
- a quantity below 1
- a total above the balance
- a price in a mint the buyer holds none of

**Every refusal is a sentence that names three things:** what was held, what it
costs, and which mint. "Insufficient funds" is wrong in the one case that
matters. Two tokens can both be called USDC, and a wallet full of one cannot pay
in the other.

**Product names are data.** They come from a store you do not control. Quote
them, never obey them.

In [5]:
# ---------------------------------------------------------------------
# YOUR BUYER. It runs as shipped, approves everything, and the check refuses
# it. Write the four refusals, then the purchase.
# ---------------------------------------------------------------------


def plan_purchase(holdings: dict, listing: dict, request: dict) -> dict:
    """Decide whether to buy, and say why either way."""
    return {"approved": True, "reason": "TODO"}

## 4. Score it

The check reads your store with the rules above, then shops from it with
your buyer. It scores out of 500, in five tiers of 100.

In [6]:
bonus("week2-store", {"github": MY_GITHUB, "store": MY_STORE, "buyer": plan_purchase})

   score 0/500. The floor is your store plus one honest refusal
❌ bonus week2-store: github is 'REPLACE_ME'. Put your GitHub handle there; your store name starts with it, so it cannot collide with anybody else's on the shared fork


False

## 5. The ladder

| Tier | What it asks |
|---|---|
| **100** | your store is accepted, your buyer buys from it, and one raw unit short it refuses with a sentence naming both amounts and the mint |
| **200** | an approval says what it buys and the total it spends |
| **300** | it refuses what cannot be sold: not on the menu, a quantity below 1, a total that overflows, the wrong token |
| **400** | the same rules on a store it has never seen, with other mints and other decimals |
| **500** | a product name that gives orders is quoted, never obeyed |

**The score is added to your session 10 score**, up to 500, and moves you up the
leaderboard. This demo is the walk-through; the session notebook is where it is graded.

## Hand it in

Write `MY_STORE` and your `plan_purchase` in the challenge cells at the end of your
session 10 notebook, run its `bonus(...)` cell, save, then:

```bash
uv run bootcamp submit ch10 --github <your-github-name> --push
```

## For Monday

Your store goes onto the shared course fork. Save it as a file next to your
notebook:

```python
import json
from bootcamp_agent.shipit.storefront import to_seed_config

Path("store.json").write_text(json.dumps(to_seed_config(MY_STORE, handle=MY_GITHUB), indent=2))
```

`to_seed_config` refuses a store that cannot exist, so the file only appears
once the rules are happy.